# 🎬 End-to-End Movie Recommendation System (ML & NLP)

Welcome to the **CineMatch Machine Learning Engine** notebook!

### 📌 Project Overview
This project builds a **Content-Based Movie Recommendation System** that analyzes narrative descriptions, genres, and taglines to discover mathematically similar films. When a user chooses a movie they love, the recommender calculates the cosine angle between the seed film's feature vector and every film in the database.

### 🧠 Core Machine Learning Workflow
1. **Data Ingestion & Exploratory Analysis (EDA)**: Understanding dataset structure, distributions, and missing fields.
2. **Data Cleaning & Filtering**: Eliminating duplicates, handling null values, and pruning irrelevant columns.
3. **JSON Parsing & Feature Extraction**: Decoding nested JSON structures in the `genres` attribute.
4. **Feature Engineering**: Merging `overview`, `genres`, and `tagline` into a unified textual representation (`tags`).
5. **Natural Language Processing (NLP)**: Lowercasing, regex punctuation cleaning, stopword removal, and WordNet lemmatization.
6. **Feature Extraction via TF-IDF**: Converting text tokens into a numerical vector space (unigrams and bigrams, up to 50,000 features).
7. **Similarity Computation**: Computing Cosine Similarity between TF-IDF document vectors.
8. **Inference Function & Verification**: Designing and evaluating the recommendation function with seed movies.
9. **Artifact Serialization**: Exporting compressed `.pkl` files to feed the production FastAPI backend and web interface.


## 1. 📦 Environment Setup & Library Imports
Import standard data science, visualization, and machine learning libraries:
- `numpy` & `pandas`: Vector operations and tabular data wrangling
- `matplotlib` & `seaborn`: Statistical data visualizations
- `warnings`: Suppressing non-critical warnings for clean outputs


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## 2. 📂 Data Ingestion & Initial Exploration (EDA)
Loading the raw dataset `movies_metadata.csv` (The Movies Dataset from Kaggle/TMDB) and inspecting its schema, columns, and initial records.


In [4]:
df = pd.read_csv('movies_metadata.csv')

In [5]:
df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [6]:
df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='object')

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

In [8]:
df.describe()

,revenue,runtime,vote_average,vote_count
count,4.546000e+04,45203.000000,45460.000000,45460.000000
mean,1.120935e+07,94.128199,5.618207,109.897338
std,6.433225e+07,38.407810,1.924216,491.310374
min,0.000000e+00,0.000000,0.000000,0.000000
25%,0.000000e+00,85.000000,5.000000,3.000000
50%,0.000000e+00,95.000000,6.000000,10.000000
75%,0.000000e+00,107.000000,6.800000,34.000000
max,2.787965e+09,1256.000000,10.000000,14075.000000


In [9]:
df.shape

(45466, 24)

In [10]:
df.isnull().sum()

,0
adult,0
belongs_to_collection,40972
budget,0
genres,0
homepage,37684
id,0
imdb_id,17
original_language,11
original_title,0
overview,954


## 3. 🧹 Data Cleaning & Duplicate Removal
Detecting and dropping duplicate movie records to ensure accurate indexing and avoid redundant recommendations.


In [11]:
df = df.drop_duplicates().reset_index(drop=True)

In [12]:
df.duplicated().sum()

np.int64(0)

## 4. 🎯 Feature Selection
Selecting the 6 core features required for content-based recommendation:
- `title`: Movie name (primary identifier)
- `overview`: Narrative plot summary
- `genres`: Film categories (Action, Drama, Sci-Fi, etc.)
- `tagline`: Marketing slogan capturing the movie's mood
- `vote_average`: User rating score
- `popularity`: Metric reflecting audience engagement


In [13]:
df = df [['title', 'overview','genres','tagline','vote_average','popularity']]

In [14]:
df.isnull().sum()

,0
title,6
overview,954
genres,0
tagline,25045
vote_average,6
popularity,5


## 5. 🛠️ Missing Value Handling & Imputation
- Dropping records missing a movie `title`
- Filling missing `overview` with an empty string
- Filling missing `tagline` with an empty string


In [16]:
df = df.dropna(subset = ['title'])

In [17]:
df['overview'] = df['overview'].fillna('')

## 6. 🎭 Genre Extraction & String Parsing
The `genres` column is stored as stringified JSON lists (e.g. `[{'id': 16, 'name': 'Animation'}, ...]`).
We use Python's `ast.literal_eval` to safely parse the dictionaries and extract only the genre names as space-separated tokens.


In [19]:
df.iloc[0]['genres']
import ast

In [20]:
df['genres'] = df['genres'].apply(lambda x: " ".join([i['name'] for i in ast.literal_eval(x)]))

In [22]:
df.head()

,title,overview,genres,tagline,vote_average,popularity
0,Toy Story,"Led by Woody, Andy's toys live happily in his ...",Animation Comedy Family,NaN,7.7,21.946943
1,Jumanji,When siblings Judy and Peter discover an encha...,Adventure Fantasy Family,Roll the dice and unleash the excitement!,6.9,17.015539
2,Grumpier Old Men,A family wedding reignites the ancient feud be...,Romance Comedy,Still Yelling. Still Fighting. Still Ready for...,6.5,11.7129
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Comedy Drama Romance,Friends are the people who let you be yourself...,6.1,3.859495
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,Comedy,Just When His World Is Back To Normal... He's ...,5.7,8.387519


In [23]:
df['tagline'] = df['tagline'].fillna('')

In [24]:
df.isnull().sum()

,0
title,0
overview,0
genres,0
tagline,0
vote_average,0
popularity,0


## 7. 🏷️ Feature Engineering: Creating Unified `tags`
Combining the `overview`, extracted `genres`, and `tagline` into a single consolidated text column called `tags`.
This unified text encapsulates the theme, narrative, and aesthetic identity of each film.


In [25]:
df['tags'] = df['overview'] + df['genres'] + df['tagline']

## 8. 🔤 Natural Language Processing (NLP) Pipeline
To extract the most informative signals from `tags`, we perform standard NLP preprocessing:
1. **Regex Cleaning**: Stripping special characters and non-alphanumeric punctuation.
2. **Lowercasing**: Normalizing case to prevent duplicate token representations.
3. **Stopword Filtering**: Removing common English filler words ('the', 'is', 'and', etc.) using NLTK.
4. **Lemmatization**: Reducing words to their root semantic lemma (e.g., 'dreams' -> 'dream', 'flying' -> 'fly') using WordNet.


In [28]:
df['tags'][1]

"When siblings Judy and Peter discover an enchanted board game that opens the door to a magical world, they unwittingly invite Alan -- an adult who's been trapped inside the game for 26 years -- into their living room. Alan's only hope for freedom is to finish the game, which proves risky as all three find themselves running from giant rhinoceroses, evil monkeys and other terrifying creatures.Adventure Fantasy FamilyRoll the dice and unleash the excitement!"

In [31]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

In [32]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [33]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [34]:
def preprocess_text(text):
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

In [35]:
df['tags'] = df['tags'].apply(preprocess_text)

In [38]:
df = df.reset_index(drop=True)

In [42]:
indices = pd.Series(df.index, index=df['title']).drop_duplicates()
indices

,0
title,
Toy Story,0
Jumanji,1
Grumpier Old Men,2
Waiting to Exhale,3
Father of the Bride Part II,4
...,...
Subdue,45442
Century of Birthing,45443
Betrayal,45444


## 9. 📐 Text Vectorization with TF-IDF
Converting the preprocessed text into high-dimensional numerical vectors using **TF-IDF (Term Frequency - Inverse Document Frequency)**:
- `max_features=50000`: Captures the 50,000 most statistically informative words and word pairs.
- `ngram_range=(1, 2)`: Captures both unigrams ('space') and bigrams ('space travel', 'time travel').
- `stop_words='english'`: Additional safety filter for language commonalities.


In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [44]:
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1,2), stop_words='english')

In [47]:
tfidf_matrix = tfidf.fit_transform(df['tags'])

In [48]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1422141 stored elements and shape (45447, 50000)>

## 10. 🔍 Cosine Similarity & Recommendation Logic
**Cosine Similarity** calculates the cosine of the angle between two multi-dimensional vectors:
$$	ext{Cosine Similarity}(A, B) = rac{A \cdot B}{\|A\| \|B\|}$$
- A score of **1.0** indicates identical content distribution.
- A score approaching **0.0** indicates entirely dissimilar themes.

The `recomend(title, n)` function looks up the seed movie's vector, computes its dot product against all catalog vectors, and returns the top $N$ closest films.


In [49]:
from sklearn.metrics.pairwise import cosine_similarity

In [56]:
def recomend(title, n=10):
  if title not in df['title'].values:
    return "Movie not found"

  idx = indices[title]
  sim_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
  similar_idx = sim_score.argsort()[::-1][1:n+1]
  return  df['title'].iloc[similar_idx]

In [62]:
recomend('Avatar')

,title
26547,Avatar 2
13883,The Inhabited Island
6414,Lara Croft Tomb Raider: The Cradle of Life
28633,Stand by Me Doraemon
2457,The Matrix
29976,Shakedown
21567,The Secret of the Third Planet
41917,The Last Dragonslayer
35681,Die Mondverschwörung
7472,Frank Herbert's Dune


## 11. 💾 Model Export & Artifact Serialization
To deploy this model to production without needing to re-vectorize thousands of documents on every server boot:
- `tfidf_matrix.pkl`: Pre-computed sparse TF-IDF feature matrix
- `indices.pkl`: Reverse-lookup index mapping movie titles to matrix row indices
- `df.pkl`: Cleaned movie metadata catalog
- `tfidf.pkl`: Trained TF-IDF vectorizer

These artifacts are loaded directly by `recommender.py` and served by the FastAPI application.


In [65]:
import pickle
pickle.dump(tfidf_matrix,open('tfidf_matrix.pkl','wb'))

pickle.dump(indices,open('indices.pkl','wb'))
df.to_pickle('df.pkl')
pickle.dump(tfidf,open('tfidf.pkl','wb'))

## 12. 🚀 Next Steps: Production Web App
The artifacts serialized above power the full-stack CineMatch application:
- **Backend**: FastAPI (`main.py`) exposes REST endpoints (`/api/search`, `/api/recommendations`, `/api/movie/{title}`) and enriches movies with high-resolution posters from the OMDb API.
- **Frontend**: A responsive web interface (`static/`) styled with a custom green palette (`#E9F5EA`, `#A6D7A8`, `#66BC6A`, `#145E1A`) offering live autocomplete, similarity scores, and movie detail modals.
- **Run the Application**:
  ```powershell
  .\.venv\Scripts\python.exe -m uvicorn main:app --reload
  ```
